# GRU Modeli — Hiperparametre Aramasi ve Egitim

**Bu notebook ne yapiyor?**
`6_gru_preprocess.ipynb`'nin hazirladigi tensorleri (`gru_arrays.npz`) okuyup, bir GRU (Gated Recurrent Unit) modeli kuruyor, `keras_tuner` ile en iyi hiperparametre kombinasyonunu ariyor, o kombinasyonla modeli sondan bastan egitiyor ve test setinde degerlendiriyor.

**Neden ayri bir notebook, `6_gru_preprocess.ipynb`'nin devami degil?**
On isleme (veri hizalama, PCA, olcekleme) ve model egitimi (keras_tuner aramasi) ayni kernel'de yapildiginda RAM tuketimi birikiyor — buyuk pandas dataframe'leri + TensorFlow'un kendi bellek kullanimi bir arada cakisinca kernel cokebiliyor (bu projede tam olarak boyle oldu). Bu notebook bagimsiz calisir: `6_gru_preprocess.ipynb`'nin kernel'ine ihtiyaci yok, her seyi diskten (`gru_arrays.npz`) okur. Dogru sira: once `6_gru_preprocess.ipynb`'yi calistirip array'leri kaydet, o kernel'i kapat, sonra bu notebook'u calistir.

**Egitim yarida kesilirse ne olur?**
`tuner` nesnesi, `directory="kt_search"` altina her trial'i diske yaziyor. Arama hucresi `overwrite=True` VERILMEDEN tekrar calistirilirsa, keras_tuner daha once tamamlanmis trial'lari tekrar calistirmiyor, kaldigi yerden devam ediyor — yani bir kernel cokmesi, o ana kadar tamamlanmis trial'lari kaybettirmiyor.


In [1]:
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from keras_tuner.tuners import RandomSearch

random.seed(46)
np.random.seed(46)
tf.random.set_seed(46)


I0000 00:00:1789921681.616021   76039 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789921681.653233   76039 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789921682.636236   76039 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. On islenmis array'leri yukle

**Neden `X_seq`'i yeniden hesaplamiyoruz, dogrudan `.npz`'den mi okuyoruz?**
`6_gru_preprocess.ipynb` zaten veri hizalama, PCA ve olcekleme gibi pahali islemleri bir kez yapip sonucu diske yazdi. Ayni islemi burada tekrarlamak hem zaman kaybi hem de o notebook'un butun bellek yukunu bu kernel'e tasimak anlamina gelirdi. `float32` olarak kaydedilmis olmasi (float64 yerine) yukleme boyutunu da yariya indiriyor.


In [2]:
data = np.load('/home/can/zero-to-ai-architect/runsight/gru_arrays.npz')
X_train_scaled = data["X_train_scaled"]
X_test_scaled = data["X_test_scaled"]
y_train = data["y_train"]
y_test = data["y_test"]
yc_train = data["yc_train"]
yc_test = data["yc_test"]

print("X_train_scaled:", X_train_scaled.shape, "X_test_scaled:", X_test_scaled.shape)


X_train_scaled: (55533, 500, 4) X_test_scaled: (13884, 500, 4)


## 2. Model tanimi

**Mimari neden bu sekilde — iki GRU katmani + dropout + tek sigmoid cikis?**
Ilk GRU katmani (`return_sequences=True`) ham 500 adimlik zaman serisini isleyip her adim icin bir temsil uretiyor; ikinci GRU katmani bu diziyi tek bir ozet vektore sikistiriyor. Ardindan tek noronlu, sigmoid aktivasyonlu bir `Dense` katmani bu ozeti [0, 1] arasinda bir olasiliga (aerobik/anaerobik) cevirir. Iki katman, kucuk veri setinde ezberlemeye yetecek kadar buyuk olmayacak, ama ogrenmeye yetecek kadar da kucuk olmayacak bir kapasite dengesi kurmak icin secildi.

**Dropout katmanlari neden her GRU'dan hemen sonra?**
Dropout, bir katmanin ciktisindaki noronlarin bir kismini rastgele sifirlayarak modelin belirli noronlara asiri baglanmasini (overfitting) engeller. Her GRU katmanindan hemen sonra konulmasi, bir sonraki katmanin (veya cikis katmaninin) onceki katmanin TUM cikisina degil, onun rastgele bir alt kumesine guvenmeyi ogrenmesini sagliyor.

**`hp.Int`, `hp.Float`, `hp.Choice` ne ise yariyor?**
Bunlar sabit degerler degil, `keras_tuner`'a "bu parametreyi su aralikta/bu secenekler arasinda dene" demenin yolu. Hangi araliklarin secildigi ve neden, bir sonraki bolumde aciklaniyor.

**Kayip fonksiyonu neden `binary_crossentropy`, metrik neden AUC (sadece accuracy degil)?**
Iki sinifli bir siniflandirma oldugu icin `binary_crossentropy` dogal secim. Metrik olarak `accuracy`'nin yaninda `AUC` de izleniyor cunku veri seti sinif dengesizligine sahip olabilir — boyle durumlarda accuracy yaniltici olabilir (model her seye cogunluk sinifini deyip yuksek accuracy alabilir), AUC ise esikten bagimsiz, modelin iki sinifi gercekten ayirt edebilme yetenegini olcer.


In [3]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])))
    model.add(GRU(
        units=hp.Int("units_1", min_value=16, max_value=64, step=16),
        return_sequences=True,
        kernel_regularizer=l2(hp.Choice("l2_1", [0.0, 1e-4, 1e-3])),
    ))
    model.add(Dropout(hp.Float("dropout_1", 0.1, 0.5, step=0.1)))
    model.add(GRU(units=hp.Int("units_2", min_value=8, max_value=32, step=8)))
    model.add(Dropout(hp.Float("dropout_2", 0.1, 0.5, step=0.1)))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=hp.Choice("lr", [1e-2, 1e-3, 1e-4])),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


## 3. Hiperparametre aramasi (`tuner.search`)

**`class_weight` neden hesaplaniyor?**
Eger `y_train` icindeki iki sinif (aerobik/anaerobik) esit sayida degilse, model varsayilan olarak cogunluk sinifina yanlilik gosterebilir. `class_weight`, azinlik sinifindaki her hataya daha fazla "ceza" agirligi vererek bu egilimi dengeler — bu bir veri kalitesi duzeltmesi degil, dogrudan egitim surecini etkileyen bir parametre.

**`objective="val_auc"` neden `val_accuracy` degil?**
Ayni gerekce: sinif dengesizliginde accuracy yaniltici olabilecegi icin, tuner'in "en iyi" dedigi kombinasyonun gercekten iki sinifi ayirt etmede iyi olmasini, sadece cogunluk sinifini ezberlememesini istiyoruz.

**Arama uzayi (units, dropout, l2, learning rate) nasil secildi?**
Kucuk bir veri setinde buyuk bir GRU (orn. 128+ birim) ezberlemeye cok yatkin olacagi icin katman boyutlari (16-64 ve 8-32) kucuk tutuldu; ikinci katman ilkinden kucuk olacak sekilde bir "huni" mimarisi hedeflendi. Dropout araligi (0.1-0.5) hafif ile agresif regularizasyon arasindaki bolgeyi tariyor. Learning rate ([1e-2, 1e-3, 1e-4]) logaritmik olcekte secildi cunku Adam optimizer'in davranisi learning rate'in buyukluk mertebesine duyarli, dogrusal farkina degil.

**`EarlyStopping` neden var, `patience=5` ne anlama geliyor?**
Bir modelin egitimi sirasinda `val_auc` 5 epoch boyunca iyilesmezse egitim erken durduruluyor ve en iyi agirliklar geri yukleniyor (`restore_best_weights=True`) — bu hem zamandan tasarruf saglar hem de modelin fazla epoch'ta ezberlemeye baslamasini engeller.

**`max_trials=10` neden bu kadar az?**
Her trial'in kendi basina epoch'lar boyunca egitim gerektirdigini dusununce (bu projede toplam arama ~5.5 saat surdu), 10 trial, sinirli zaman icinde makul sayida kombinasyonu denemek ile pratik sure arasinda bir denge.


In [4]:
n0, n1 = np.bincount(y_train)
class_weight = {0: len(y_train) / (2 * n0), 1: len(y_train) / (2 * n1)}
print("class_weight:", class_weight)

early_stop = EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)

tuner = RandomSearch(
    build_model,
    objective="val_auc",
    max_trials=10,
    executions_per_trial=1,
    directory="kt_search",
    project_name="gru_aerobic_anaerobic",
    seed=46,
)

tuner.search(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1,
)


Trial 10 Complete [00h 41m 11s]
val_auc: 0.7606422305107117

Best val_auc So Far: 0.8464333415031433
Total elapsed time: 05h 22m 23s


## 4. En iyi modeli egit ve degerlendir

**`tuner.search` zaten modelleri egitmisti, neden tekrar egitiyoruz?**
`tuner.search`, her kombinasyonu SADECE birbirleriyle kiyaslamak icin kisa epoch sayisiyla (burada 30) calistirir; amac en iyisini bulmaktir, o modeli nihai olarak kullanmak degil. En iyi hiperparametreler belirlendikten sonra, o mimariyi bastan, daha uzun (50 epoch, yine `EarlyStopping` ile korunarak) egitip asil kullanilacak modeli elde ediyoruz.

**`gc.collect()` ve `tf.keras.backend.clear_session()` neden burada?**
Arama sirasinda 10 farkli model art arda olusturulup egitildigi icin TensorFlow'un ic hesaplama grafiginde biriken bellek kalintilarini temizliyoruz — aksi halde nihai egitim, gereksiz yere daha fazla RAM kullanarak baslar.


In [5]:
gc.collect()
tf.keras.backend.clear_session()

best_hp = tuner.get_best_hyperparameters(1)[0]
print("en iyi hiperparametreler:", best_hp.values)

best_model = tuner.hypermodel.build(best_hp)
history = best_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1,
)


en iyi hiperparametreler: {'units_1': 64, 'l2_1': 0.0, 'dropout_1': 0.1, 'units_2': 16, 'dropout_2': 0.5, 'lr': 0.001}
Epoch 1/50
695/695 ━━━━━━━━━━━━━━━━━━━━ 154s 219ms/step - accuracy: 0.6312 - auc: 0.6777 - loss: 0.6449 - val_accuracy: 0.4949 - val_auc: 0.4926 - val_loss: 0.6942
Epoch 2/50
695/695 ━━━━━━━━━━━━━━━━━━━━ 152s 219ms/step - accuracy: 0.6445 - auc: 0.6949 - loss: 0.6345 - val_accuracy: 0.6908 - val_auc: 0.7457 - val_loss: 0.5957
Epoch 3/50
695/695 ━━━━━━━━━━━━━━━━━━━━ 147s 211ms/step - accuracy: 0.6772 - auc: 0.7183 - loss: 0.6177 - val_accuracy: 0.6922 - val_auc: 0.7431 - val_loss: 0.5984
Epoch 4/50
695/695 ━━━━━━━━━━━━━━━━━━━━ 145s 209ms/step - accuracy: 0.6511 - auc: 0.6907 - loss: 0.6376 - val_accuracy: 0.6549 - val_auc: 0.7381 - val_loss: 0.6287
Epoch 5/50
695/695 ━━━━━━━━━━━━━━━━━━━━ 145s 208ms/step - accuracy: 0.5590 - auc: 0.5939 - loss: 0.6839 - val_accuracy: 0.5483 - val_auc: 0.7186 - val_loss: 0.6877
Epoch 6/50
695/695 ━━━━━━━━━━━━━━━━━━━━ 149s 214ms/step - acc

## 5. Test setinde degerlendirme

**Neden sadece `classification_report` degil, ayrica `roc_auc_score` ve confusion matrix de bakiliyor?**
`classification_report`, precision/recall/f1'i sinif bazinda gosterir — hangi sinifta modelin daha zayif oldugunu gormek icin gerekli. `roc_auc_score`, egitim sirasinda izlenen `val_auc` metriginin, daha once hic gormedigi test verisinde de gecerli olup olmadigini dogrular. Confusion matrix ise sayisal ozetin arkasindaki gercek hata dagilimini (kac tane aerobik anaerobik sanildi, kac tane anaerobik aerobik sanildi) gosterir — iki tur hatanin (false positive / false negative) bu projede esit derecede onemli olup olmadigina karar vermek icin faydali.

**Esik neden 0.5?**
`y_pred_proba >= 0.5`, modelin urettigi olasiligi ikili karara cevirmenin en dogal varsayilan yolu. `class_weight` egitim sirasinda zaten sinif dengesizligini telafi ettigi icin, tahmin asamasinda esigi kaydirmaya (orn. 0.4 veya 0.6) gerek kalmiyor — ama eger bir sinifin yanlis siniflandirilmasi digerinden daha maliyetliyse, bu esik sonradan ayarlanabilir.


In [6]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred_proba = best_model.predict(X_test_scaled).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["aerobik", "anaerobik"]))
print("test AUC:", roc_auc_score(y_test, y_pred_proba))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))


434/434 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step
              precision    recall  f1-score   support

     aerobik       0.78      0.73      0.76      6980
   anaerobik       0.75      0.79      0.77      6904

    accuracy                           0.76     13884
   macro avg       0.76      0.76      0.76     13884
weighted avg       0.76      0.76      0.76     13884

test AUC: 0.8503817603349413
confusion matrix:
 [[5114 1866]
 [1422 5482]]


## 6. Baseline ile karsilastir

**`training_zone` (yani `y_test`) zaten `run_zone_pipeline`'in kendi ciktisi degil mi? O zaman GRU'yu ona karsi kiyaslamak neyi gosterir?**
Burada dikkat edilmesi gereken onemli bir nokta var: `y_test`, GRU'nun egitildigi ETIKETIN ta kendisi — yani GRU'nun tahminini `y_test`'e karsi "dogruluk" olarak olcmek zaten yapildi (bir onceki bolumdeki `classification_report`). Baseline'i ayri bir model gibi kurup ayni etikete karsi kiyaslamak tautoloji olurdu (esik mantigi zaten etiketin kendisi).

**Peki burada asil sorulmasi gereken soru ne?**
GRU'ya kisinin kisisel nabiz esigi (Karvonen HR reserve orani) hic verilmedi — sadece ham speed/heart_rate/altitude/GPS kanallarini gordu. Sorulmasi gereken: GRU, bu esik bilgisine sahip olmadan, esik-tabanli sistemin urettigi SUREKLI skorla (`anaerobic_time_frac`, `yc_test`) ne kadar ortusen bir cikti uretebiliyor? GRU'nun tahmin olasiligi (`y_pred_proba`) ile `yc_test` arasindaki korelasyona bakmak, GRU'nun ham sinyallerden esige benzer bir efor-yogunlugu kavramini kendi basina yakalayip yakalamadigini gosterir — bu, modelin etiketi ezberlemek yerine gercekten fizyolojik olarak anlamli bir sey ogrenip ogrenmedigine dair bagimsiz bir kanit sayilir.


## 7. Modeli kaydet

**Model neden `.keras` formatinda kaydediliyor?**
`.keras`, mimariyi, agirliklari ve optimizer durumunu tek bir dosyada tasiyan Keras'in kendi kayit formati — modeli daha sonra tek satirla (`load_model`) eksiksiz geri yuklemeyi saglar.

**Bu dosya tek basina yeterli mi?**
Hayir. Modelin girdisi, `6_gru_preprocess.ipynb`'de kaydedilen `gru_preprocessing.pkl` icindeki `MinMaxScaler`'lar ile ayni sekilde olceklenmis olmali. Yani gercek kullanimda (orn. serverless inference) hem `gru_model.keras` hem de `gru_preprocessing.pkl` birlikte tasinmali — biri olmadan digeri anlamli tahmin uretemez.


In [7]:
best_model.save('/home/can/zero-to-ai-architect/runsight/gru_model.keras')
print("model kaydedildi.")


model kaydedildi.
